In [1]:
from pandas.core.interchange.dataframe_protocol import DataFrame

from _bootstrap import setup

import cv2
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from enum import StrEnum
import os
import pandas as pd

In [2]:
class OrientacionCorte(StrEnum):
    VERTICAL = "vertical"
    HORIZONTAL = "horizontal"

In [3]:
def obtener_o_crear_directorio(ruta_base: Path, carpeta_destino: str):
    ruta_completa = ruta_base / carpeta_destino

    ruta_completa.mkdir(parents=True, exist_ok=True)

    if not ruta_completa.exists():
        raise RuntimeError("No se pudo crear el directorio")

    return ruta_completa


# Funciones
def obtener_nombre_carpeta_clase(valor_clase: str | int | float):
    """
    Devuelve el nombre de la carpeta según la 'Clase Ana':
    - Si hay número → 'clase01', 'clase02', ... 'clase11', etc.
    - Si está vacío/NaN → 'sin clase'
    """
    if pd.isna(valor_clase) or str(valor_clase).strip() == "":
        return "sin_clase"
    try:
        numero = int(float(valor_clase))  # maneja '11.0' como 11
        return f"clase{numero:02d}"
    except ValueError:
        return "sin_clase"



In [4]:
def obtener_rutas_perfiles(ruta_origen: Path, clase: int | str, extension: str) -> list[Path]:
    nombre_carpeta = obtener_nombre_carpeta_clase(clase)
    carpeta = ruta_origen / nombre_carpeta

    return list(carpeta.glob(f"*.{extension}"))

In [5]:
def determinar_orientacion_corte(clase: str) -> OrientacionCorte:
    if clase in range(1, 10):
        return OrientacionCorte.VERTICAL
    return OrientacionCorte.HORIZONTAL


def crear_ruta_archivo(ruta_base: Path, nombre_archivo: str, sufijo: str):
    return os.path.join(ruta_base, f"{nombre_archivo}{sufijo}.png")

In [6]:
def mostrar_imagen(imagen):
    plt.imshow(imagen, cmap=plt.cm.gray)
    plt.axis("on")
    plt.figure()


def obtener_nombre_archivo(ruta: Path) -> str:
    return Path(ruta).stem


def guardar_imagen(ruta: Path, imagen):
    cv2.imwrite(ruta, imagen)


def leer_imagen(ruta_imagen: str):
    img = cv2.imread(ruta_imagen, cv2.IMREAD_GRAYSCALE)
    return img


def cortar_imagen(img, orientacion_corte: OrientacionCorte):
    alto, ancho = img.shape
    # Trabajamos sobre un lienzo cuadrado para conservar el tamaño final en ambas mitades.
    lado_lienzo = max(alto, ancho)

    # Desplazamientos para centrar la imagen original dentro del lienzo cuadrado.
    offset_y = (lado_lienzo - alto) // 2
    offset_x = (lado_lienzo - ancho) // 2

    # Cortes en coordenadas de la imagen original (fuente).
    corte_tope_src, corte_base_src = obtener_cortes_fuente(alto, ancho, orientacion_corte)

    imagen_tope = crear_lienzo_cuadrado(img, lado_lienzo)
    imagen_base = crear_lienzo_cuadrado(img, lado_lienzo)

    # Mapeamos los cortes fuente a las coordenadas del lienzo centrado.
    corte_tope_dst = trasladar_corte_a_lienzo(corte_tope_src, offset_y, offset_x, lado_lienzo)
    corte_base_dst = trasladar_corte_a_lienzo(corte_base_src, offset_y, offset_x, lado_lienzo)

    # Copiamos solo la zona correspondiente a cada mitad.
    imagen_tope[corte_tope_dst] = img[corte_tope_src]
    imagen_base[corte_base_dst] = img[corte_base_src]

    return imagen_tope, imagen_base


def obtener_cortes_fuente(alto: int, ancho: int, orientacion_corte: OrientacionCorte, ) -> \
        tuple[tuple[slice, slice], tuple[slice, slice]]:
    # VERTICAL: divide en mitad superior/inferior.
    if orientacion_corte == OrientacionCorte.VERTICAL:
        return (
            (slice(0, alto // 2), slice(0, ancho)),
            (slice(alto // 2, alto), slice(0, ancho)),
        )

    # HORIZONTAL: divide en mitad izquierda/derecha.
    return (
        (slice(0, alto), slice(0, ancho // 2)),
        (slice(0, alto), slice(ancho // 2, ancho)),
    )


def crear_lienzo_cuadrado(img, lado: int):
    return np.full((lado, lado), 255, dtype=img.dtype)


def trasladar_corte_a_lienzo(
        corte_src: tuple[slice, slice],
        offset_y: int,
        offset_x: int,
        lado_lienzo: int,
) -> tuple[slice, slice]:
    # Separar ejes permite desplazar cada dimensión con su offset correspondiente.
    slice_y, slice_x = corte_src

    # Devuelve el mismo recorte, pero expresado en coordenadas del lienzo centrado.
    return (
        trasladar_slice(slice_y, offset_y, lado_lienzo),
        trasladar_slice(slice_x, offset_x, lado_lienzo),
    )


def trasladar_slice(segmento: slice, offset: int, limite: int) -> slice:
    # Ajusta el inicio y fin del segmento al sistema de coordenadas del lienzo.
    inicio = offset if segmento.start is None else segmento.start + offset
    fin = limite if segmento.stop is None else segmento.stop + offset
    return slice(inicio, fin)

In [7]:
def procesar_perfil(
        ruta_perfil: Path,
        orientacion: OrientacionCorte,
        ruta_destino_clase: Path
) -> pd.DataFrame:
    try:
        img = leer_imagen(ruta_perfil)
        tope, base = cortar_imagen(img, orientacion)
        nombre_original = obtener_nombre_archivo(ruta_perfil)

        ruta_tope = crear_ruta_archivo(ruta_destino_clase, nombre_original, "_tope")
        ruta_base = crear_ruta_archivo(ruta_destino_clase, nombre_original, "_base")

        guardar_imagen(ruta=ruta_tope, imagen=tope)
        guardar_imagen(ruta=ruta_base, imagen=base)

        return pd.DataFrame([
            {
                "archivo_origen": ruta_perfil.name,
                "archivo_generado": Path(ruta_tope).name,
                "mitad": "tope",
            },
            {
                "archivo_origen": ruta_perfil.name,
                "archivo_generado": Path(ruta_base).name,
                "mitad": "base",
            },
        ])

    except Exception as e:
        print(f"Error procesando {ruta_perfil}: {e}")
        return pd.DataFrame()


def procesar_clase(ruta_origen: Path, ruta_destino: Path, clase: int | str) -> pd.DataFrame:
    dfs: list[pd.DataFrame] = []

    perfiles = obtener_rutas_perfiles(ruta_origen, clase, extension="png")

    if not perfiles:
        return pd.DataFrame()

    orientacion = determinar_orientacion_corte(clase)

    ruta_destino_clase = obtener_o_crear_directorio(
        ruta_destino,
        obtener_nombre_carpeta_clase(clase)
    )

    for ruta_perfil in perfiles:
        df_perfil = procesar_perfil(ruta_perfil, orientacion, ruta_destino_clase)
        dfs.append(df_perfil)

    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


def procesar_imagenes(ruta_origen: Path, ruta_destino: Path, clases: list[int | str]) -> pd.DataFrame:

    dfs: list[pd.DataFrame] = []

    for clase in clases:
        df_clase = procesar_clase(ruta_origen, ruta_destino, clase)
        dfs.append(df_clase)

    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


In [8]:
def construir_df_desde_mitades(
    df_completos: pd.DataFrame,
    df_mitades: pd.DataFrame,
    how: str = "inner",  # usa "left" para conservar todas las mitades aunque no encuentren match
) -> pd.DataFrame:
    """
    Crea un nuevo dataframe a partir de df_completos y df_mitades:
    - Join por: df_completos.Identificador == stem(df_mitades.archivo_origen)
    - El nuevo 'Identificador' toma el valor de df_mitades.archivo_generado
    - Conserva el resto de columnas de df_completos
    - Agrega columnas: 'archivo_origen' y 'mitad' desde df_mitades
    """
    # Nos quedamos solo con las columnas necesarias del dataframe de mitades
    mitades = df_mitades[["archivo_origen", "archivo_generado", "mitad"]].copy()
    mitades["archivo_origen_stem"] = mitades["archivo_origen"].map(lambda nombre: Path(nombre).stem)

    # Join entre ambos dataframes usando el nombre base del archivo original (sin extensión)
    df_join = mitades.merge(
        df_completos,
        left_on="archivo_origen_stem",
        right_on="Identificador",
        how=how,
        validate="many_to_one",  # muchas mitades pueden venir de un mismo original
    )

    # El nuevo Identificador pasa a ser el stem del archivo generado (sin extensión)
    df_join["Identificador"] = df_join["archivo_generado"].map(lambda nombre: Path(nombre).stem)

    # Estructura final: mismas columnas que df_completos + archivo_origen + mitad
    columnas_finales = list(df_completos.columns) + ["archivo_origen", "mitad"]
    return df_join[columnas_finales]

In [9]:
from _bootstrap import setup

clases_a_clasificar = [1,2,3,4,5,6,7,8,9,10,11]
PATHS = setup()

csv_origen: Path = PATHS.DATA_CLASIFICADOS / "metadata.csv"
csv_destino: Path = PATHS.DATA_MITADES / "metadata.csv"

df_completos = pd.read_csv(csv_origen)

df_mitades = procesar_imagenes(PATHS.DATA_CLASIFICADOS, PATHS.DATA_MITADES, clases_a_clasificar)

df_final = construir_df_desde_mitades(df_completos, df_mitades)
df_final.to_csv(csv_destino, index=False)
df_final.head()

,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia,Ubicacion,archivo_origen,mitad
0,mb_03_tope,No,IV,12,B,7,4,1.0,188,47,464,ibero,Marroquies Bajos,mb_03.png,tope
1,mb_03_base,No,IV,12,B,7,4,1.0,188,47,464,ibero,Marroquies Bajos,mb_03.png,base
2,ba_208_3_tope,Si,IV,7,B,5,4,1.0,60,17,112,ibero,Baza,ba_208_3.png,tope
3,ba_208_3_base,Si,IV,7,B,5,4,1.0,60,17,112,ibero,Baza,ba_208_3.png,base
4,ba_208_2_tope,Si,IV,11,B,7,4,1.0,60,15,90,ibero,Baza,ba_208_2.png,tope
